In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter


df = pd.read_csv("../data/clean_faculty_data.csv")
TOTAL = len(df)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   name            112 non-null    str  
 1   profile         112 non-null    str  
 2   education       110 non-null    str  
 3   phone           112 non-null    str  
 4   address         112 non-null    str  
 5   email           112 non-null    str  
 6   specialization  112 non-null    str  
 7   personal_links  112 non-null    str  
 8   bio             112 non-null    str  
 9   teaching        112 non-null    str  
 10  research_areas  112 non-null    str  
 11  publications    112 non-null    str  
 12  embedding_text  112 non-null    str  
dtypes: str(13)
memory usage: 11.5 KB


In [3]:
def to_list(x):
    """Safely convert string-list to Python list"""
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []


def is_missing(x):
    if isinstance(x, list):
        return len(x) == 0
    if isinstance(x, str):
        return x.strip() in ["Not Provided", "Not Available", "0", ""]
    return pd.isna(x)
df["specialization"] = df["specialization"].apply(to_list)
df["teaching"] = df["teaching"].apply(to_list)
df["publications"] = df["publications"].apply(to_list)


## Missing Values Analysis

In [4]:
null_stats = []

for col in df.columns:
    missing_count = df[col].apply(is_missing).sum()
    percent = round((missing_count / TOTAL) * 100, 2)

    null_stats.append({
        "column": col,
        "missing_count": missing_count,
        "missing_percentage": percent
    })

null_df = pd.DataFrame(null_stats)

null_df


,column,missing_count,missing_percentage
0,name,0,0.00
1,profile,0,0.00
2,education,2,1.79
3,phone,34,30.36
4,address,35,31.25
5,email,1,0.89
6,specialization,0,0.00
7,personal_links,65,58.04
8,bio,43,38.39
9,teaching,3,2.68


## PhD vs Non-PhD Distribution

In [5]:
def has_phd(edu):
    if pd.isna(edu):
        return False
    return bool(re.search(r"\bph\.?d\b", str(edu).lower()))

df["has_phd"] = df["education"].apply(has_phd)

phd_count = df["has_phd"].sum()
non_phd_count = TOTAL - phd_count

phd_stats = pd.DataFrame({
    "category": ["PhD", "Non-PhD"],
    "count": [phd_count, non_phd_count],
    "percentage": [
        round(phd_count / TOTAL * 100, 2),
        round(non_phd_count / TOTAL * 100, 2)
    ]
})

phd_stats


,category,count,percentage
0,PhD,95,84.82
1,Non-PhD,17,15.18


## Unique Specializations

In [6]:
all_specializations = set()

for specs in df["specialization"]:
    for s in specs:
        all_specializations.add(s)

print("Total Unique Specialization",len(all_specializations))

Total Unique Specialization 349


## Publications Statistics

In [7]:
# Ensure publications is a list
df["publications"] = df["publications"].apply(to_list)

# Publication count per faculty
df["publication_count"] = df["publications"].apply(len)

# Statistics
avg_pubs = round(df["publication_count"].mean(), 2)
min_pubs = int(df["publication_count"].min())
max_pubs = int(df["publication_count"].max())

# Info strings (README / console friendly)
print(f"Average publications per faculty : {avg_pubs}")
print(f"Minimum publications per faculty : {min_pubs}")
print(f"Maximum publications per faculty : {max_pubs}")


Average publications per faculty : 7.41
Minimum publications per faculty : 0
Maximum publications per faculty : 50


## Teaching vs Research Information Availability

In [8]:
teaching_available = df["teaching"].apply(lambda x: len(x) > 0).sum()
research_available = df["research_areas"].apply(
    lambda x: not is_missing(x)
).sum()

availability_df = pd.DataFrame({
    "category": ["Teaching Info Available", "Research Areas Available"],
    "count": [teaching_available, research_available],
    "percentage": [
        round(teaching_available / TOTAL * 100, 2),
        round(research_available / TOTAL * 100, 2)
    ]
})

availability_df


,category,count,percentage
0,Teaching Info Available,109,97.32
1,Research Areas Available,19,16.96


## Most Common Specialization(s) and Top 10 Specializations by Faculty Count

In [9]:
all_specializations = []

for specs in df["specialization"]:
    all_specializations.extend(specs)

specialization_counts = Counter(all_specializations)

max_specialization_count = max(specialization_counts.values())
top_specializations = [
    spec for spec, cnt in specialization_counts.items()
    if cnt == max_specialization_count
]
print(f"🎯 Most common specialization(s) appear {max_specialization_count} times:")
for spec in top_specializations:
    print(f" - {spec}")

    
top_10_specializations = specialization_counts.most_common(10)

top_specializations_df = pd.DataFrame(
    top_10_specializations,
    columns=["specialization", "faculty_count"]
)

top_specializations_df


🎯 Most common specialization(s) appear 7 times:
 - Machine Learning
 - Computer Vision


,specialization,faculty_count
0,Machine Learning,7
1,Computer Vision,7
2,Information Retrieval,6
3,Image Processing,5
4,Natural Language Processing,5
5,Not Available,4
6,Signal Processing,3
7,Algorithms,3
8,Photography,3
9,Pattern Recognition,2
